# Chess RL vs Stockfish

In [ ]:
!apt-get update && apt-get install -y stockfish -qq
!pip install python-chess torch numpy matplotlib -q

In [ ]:
import torch, torch.nn as nn, torch.optim as optim, chess, chess.engine, chess.pgn, numpy as np, json
from pathlib import Path
import random, matplotlib.pyplot as plt
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = Path('/content/drive/MyDrive/chess_stockfish')
drive_path.mkdir(parents=True, exist_ok=True)

In [ ]:
class BoardEncoder:
    def __init__(self, history_size=8):
        self.history_size = history_size
    def encode(self, board):
        tensor = torch.zeros(112, 8, 8, dtype=torch.float32)
        for sq in range(64):
            p = board.piece_at(sq)
            if p:
                r, c = sq // 8, sq % 8
                idx = (p.piece_type - 1) + (0 if p.color == chess.WHITE else 6)
                tensor[idx, r, c] = 1.0
        if board.turn == chess.BLACK: tensor[12, :, :] = 1.0
        return tensor
encoder = BoardEncoder()

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, f):
        super().__init__()
        self.c1 = nn.Conv2d(f, f, 3, padding=1)
        self.b1 = nn.BatchNorm2d(f)
        self.c2 = nn.Conv2d(f, f, 3, padding=1)
        self.b2 = nn.BatchNorm2d(f)
    def forward(self, x):
        r = x
        o = torch.relu(self.b1(self.c1(x)))
        o = self.b2(self.c2(o))
        return torch.relu(o + r)
class ChessNetLc0(nn.Module):
    def __init__(self, f=128, b=6):
        super().__init__()
        self.ic = nn.Conv2d(112, f, 3, padding=1)
        self.ib = nn.BatchNorm2d(f)
        self.rb = nn.Sequential(*[ResidualBlock(f) for _ in range(b)])
        self.pc = nn.Conv2d(f, 64, 3, padding=1)
        self.pb = nn.BatchNorm2d(64)
        self.pf = nn.Linear(64*64, 4672)
        self.vc = nn.Conv2d(f, 32, 1)
        self.vb = nn.BatchNorm2d(32)
        self.vf1 = nn.Linear(32*64, 128)
        self.vf2 = nn.Linear(128, 1)
        self.wc = nn.Conv2d(f, 32, 1)
        self.wb = nn.BatchNorm2d(32)
        self.wf1 = nn.Linear(32*64, 128)
        self.wf2 = nn.Linear(128, 3)
    def forward(self, x):
        x = torch.relu(self.ib(self.ic(x)))
        x = self.rb(x)
        p = torch.relu(self.pb(self.pc(x)))
        p = p.view(p.size(0), -1)
        pol = torch.softmax(self.pf(p), dim=1)
        v = torch.relu(self.vb(self.vc(x)))
        v = v.view(v.size(0), -1)
        v = torch.relu(self.vf1(v))
        val = torch.tanh(self.vf2(v))
        w = torch.relu(self.wb(self.wc(x)))
        w = w.view(w.size(0), -1)
        w = torch.relu(self.wf1(w))
        wdl = torch.softmax(self.wf2(w), dim=1)
        return pol, val, wdl
net = ChessNetLc0().to(device)

In [ ]:
def select_move(b, n, e):
    m = list(b.legal_moves)
    if not m: return None
    sc = []
    for mv in m:
        b.push(mv)
        t = e.encode(b).unsqueeze(0).to(device)
        with torch.no_grad():
            _, v, _ = n(t)
            s = v.item()
        if b.is_checkmate(): s += 100
        elif b.is_check(): s += 10
        if b.piece_at(mv.to_square): s += 3
        b.pop()
        sc.append(s)
    return m[np.argmax(sc)]
class SF:
    def __init__(self, d=1):
        self.e = chess.engine.SimpleEngine.popen_uci('/usr/games/stockfish')
        self.d = d
    def move(self, b):
        try: return self.e.play(b, chess.engine.Limit(depth=self.d)).move
        except: return random.choice(list(b.legal_moves))
    def set_d(self, d): self.d = d
    def quit(self): self.e.quit()

In [ ]:
def play(n, e, sf, w=True):
    b = chess.Board()
    gm = []
    ml = []
    for _ in range(200):
        if b.is_game_over():
            r = b.result()
            win = (0 if r == '1-0' else 1 if r == '0-1' else 2)
            if not w: win = 1 if win == 0 else (0 if win == 1 else 2)
            return win, gm, str(chess.pgn.Game())
        is_n = (b.turn == chess.WHITE) if w else (b.turn == chess.BLACK)
        bt = e.encode(b)
        mv = select_move(b, n, e) if is_n else sf.move(b)
        if not mv: break
        b.push(mv)
        ml.append(mv)
        if is_n: gm.append({'b': bt})
    return 2, gm, str(chess.pgn.Game())
def train(n, o, gm, w, lr=0.001):
    if not gm: return 0.0
    n.train()
    tl = 0.0
    for i in range(0, len(gm), 16):
        bb = torch.stack([m['b'] for m in gm[i:i+16]]).to(device)
        p, v, wdl = n(bb)
        vt = 1.0 if w in [0,1] else (-0.3 if w == 2 else -1.0)
        wt = 0.5 if w in [0,1] else (3.0 if w == 2 else 2.0)
        vts = torch.tensor([[vt]]*len(bb), dtype=torch.float32).to(device)
        wdlt = torch.tensor([[1,0,0]] if w in [0,1] else [[0,1,0]], dtype=torch.float32).to(device).expand(len(bb),-1)
        vl = torch.mean((v - vts)**2) * wt
        wl = -torch.sum(wdlt * torch.log(wdl + 1e-8)) / len(bb)
        l = vl + wl
        o.zero_grad()
        l.backward()
        torch.nn.utils.clip_grad_norm_(n.parameters(), 1.0)
        o.step()
        tl += l.item()
    n.eval()
    return tl / max(1, len(gm) // 16)

In [ ]:
def save(n, o, ep, d, p):
    c = {'e': ep, 'd': d, 'm': n.state_dict(), 'o': o.state_dict()}
    torch.save(c, p / f'c{ep}d{d}.pt')
def load(path):
    c = torch.load(path, map_location=device)
    n = ChessNetLc0().to(device)
    o = optim.Adam(n.parameters(), 0.001)
    n.load_state_dict(c['m'])
    o.load_state_dict(c['o'])
    return n, o, c['e'], c['d']

In [ ]:
class S:
    def __init__(self, p):
        self.f = p / 's.json'
        self.s = {'d': 1, 'g': {}, 'ag': 0, 'w': 0, 'd': 0, 'l': 0, 'h': [], 'ls': []}
        if self.f.exists():
            with open(self.f) as f: self.s = json.load(f)
    def r(self, win, d, loss):
        self.s['ag'] += 1
        self.s['ls'].append(loss)
        self.s['h'].append('W' if win in [0,1] else ('D' if win == 2 else 'L'))
        ds = str(d)
        if ds not in self.s['g']: self.s['g'][ds] = {'w': 0, 'd': 0, 'l': 0}
        if win in [0,1]: self.s['g'][ds]['w'] += 1
        elif win == 2: self.s['g'][ds]['d'] += 1
        else: self.s['g'][ds]['l'] += 1
    def wr(self, d):
        ds = str(d)
        if ds not in self.s['g']: return 0.0
        g = self.s['g'][ds]
        t = g['w'] + g['d'] + g['l']
        return g['w'] / t if t > 0 else 0.0
    def save(self):
        with open(self.f, 'w') as f: json.dump(self.s, f)
    def pr(self, d):
        ds = str(d)
        if ds not in self.s['g']: return
        g = self.s['g'][ds]
        t = g['w'] + g['d'] + g['l']
        if t == 0: return
        al = np.mean(self.s['ls'][-20:]) if self.s['ls'] else 0
        print(f'D{d} W:{g["w"]:2d}({100*g["w"]/t:5.1f}%) D:{g["d"]:2d} L:{g["l"]:2d} L:{al:.4f}')
st = S(drive_path)

In [ ]:
ckpt = sorted(drive_path.glob('c*d*.pt'))
ep = ckpt[0].stem.split('d')[0][1:] if ckpt else 0
cd = int(ckpt[0].stem.split('d')[1]) if ckpt else 1
net = (load(ckpt[-1])[0] if ckpt else ChessNetLc0().to(device))
opt = optim.Adam(net.parameters(), 0.001)
sf = SF(cd)
net.eval()
print(f'Depth: {cd}\n')
for _ in range(500):
    w, gm, _ = play(net, encoder, sf, ep % 2 == 0)
    loss = train(net, opt, gm, w)
    st.r(w, cd, loss)
    ep += 1
    if ep % 10 == 0:
        print(f'Ep {ep} | ', end='')
        st.pr(cd)
    if ep % GAMES_PER_LEVEL == 0 and st.wr(cd) >= 0.6:
        cd += 1
        sf.set_d(cd)
        print(f'\nLVL UP {cd-1} -> {cd}\n')
    if ep % 20 == 0:
        save(net, opt, ep, cd, drive_path)
        st.save()
sf.quit()